# CEG-WM Colab 完整工作流执行指南

本 Notebook 在 Google Colab 上执行完整的机器学习工作流（embed/detect/calibrate/evaluate），生成新的 run_root 目录并下载最终结果。

**执行流水线**：
- **Embed 阶段**：对输入进行特征提取和防水印嵌入
- **Detect 阶段**：检测防水印信号
- **Calibrate 阶段**：校准检测阈值和概率
- **Evaluate 阶段**：评估整个工作流的性能

## 第 1 步：清理旧代码并拉取 GitHub 项目

从 GitHub 拉取 CEG-WM 代码到 Colab 工作目录。

**方式：自动拉取指定仓库与分支**
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：main

In [ ]:
import sys
import os
from pathlib import Path
import psutil

# 检测 Google Colab 环境
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("初始化 Notebook 环境")
print("=" * 80)

# 设置工作目录（按需固定到 /content，避免 /tmp 被系统回收）
if IN_COLAB:
    WORK_DIR = Path("/content")
    print(f"✓ 检测到 Google Colab 环境")
else:
    WORK_DIR = Path.cwd()
    print(f"✓ 本地 Jupyter 环境")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"  工作目录：{WORK_DIR}")

# 显示系统信息
print("\n系统信息：")
print(f"  操作系统：{sys.platform}")
print(f"  Python 版本：{sys.version.split()[0]}")
print(f"  CPU 核心数：{psutil.cpu_count()}")
print(f"  总内存：{psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"  可用内存：{psutil.virtual_memory().available / (1024**3):.1f} GB")

print("\n✓ 环境初始化完成")

In [ ]:
from pathlib import Path

print("=" * 80)
print("挂载 Google Drive")
print("=" * 80)

DRIVE_MOUNT_POINT = Path("/content/drive")
GOOGLE_DRIVE_ROOT = None
GOOGLE_DRIVE_EXPORT_DIR = None

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
        GOOGLE_DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
        GOOGLE_DRIVE_EXPORT_DIR = GOOGLE_DRIVE_ROOT / "CEG_WM_Outputs" / "Paper_Full_Cuda"
        GOOGLE_DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"✓ Google Drive 已挂载：{GOOGLE_DRIVE_ROOT}")
        print(f"✓ 导出同步目录：{GOOGLE_DRIVE_EXPORT_DIR}")
    except Exception as exc:
        GOOGLE_DRIVE_ROOT = None
        GOOGLE_DRIVE_EXPORT_DIR = None
        print(f"✗ Google Drive 挂载失败：{exc}")
else:
    print("ℹ 当前不是 Google Colab 环境，跳过 Google Drive 挂载")

In [ ]:
import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "main"
TARGET_DIR = WORK_DIR / "CEG-WM"

print("=" * 80)
print("同步 GitHub 上的 CEG-WM 代码并清理 outputs")
print("=" * 80)
print(f"目标仓库：{REPO_URL}")
print(f"目标分支：{REPO_BRANCH}")
print(f"工作目录：{WORK_DIR}")

if shutil.which("git") is None:
    raise RuntimeError("当前环境未检测到 git 命令，请先安装 git 后重试")

WORK_DIR.mkdir(parents=True, exist_ok=True)


def _print_command_tail(label: str, text: str, max_lines: int = 20) -> None:
    print(f"\n{label}（最后 {max_lines} 行）：")
    lines = (text or "").splitlines()
    print("\n".join(lines[-max_lines:]) if lines else "<empty>")


def _run_checked_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.returncode != 0:
        _print_command_tail("STDOUT", result.stdout)
        _print_command_tail("STDERR", result.stderr)
        raise RuntimeError(
            f"命令执行失败，return_code={result.returncode}，command={' '.join(command)}"
        )
    return result


sync_mode = "git_clone"
remote_url = ""
repo_exists = TARGET_DIR.exists() and (TARGET_DIR / ".git").exists()

print("\n[1/4] 同步仓库代码...")
if repo_exists:
    remote_result = subprocess.run(
        ["git", "-C", str(TARGET_DIR), "remote", "get-url", "origin"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    remote_url = remote_result.stdout.strip() if remote_result.returncode == 0 else ""
    if remote_url and remote_url.rstrip("/") != REPO_URL.rstrip("/"):
        print(f"  检测到 origin 不匹配，删除后重新拉取：{remote_url}")
        shutil.rmtree(TARGET_DIR)
        repo_exists = False

if repo_exists:
    sync_mode = "git_fetch_reset"
    print(f"  复用现有仓库：{TARGET_DIR}")
    _run_checked_command(["git", "-C", str(TARGET_DIR), "fetch", "origin", REPO_BRANCH, "--depth", "1"])
    _run_checked_command(["git", "-C", str(TARGET_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"])
    _run_checked_command(["git", "-C", str(TARGET_DIR), "reset", "--hard", "FETCH_HEAD"])
    _run_checked_command(["git", "-C", str(TARGET_DIR), "clean", "-fd"])
    print("  ✓ 仓库已更新到远端最新提交")
else:
    if TARGET_DIR.exists():
        print(f"  检测到非 Git 目录，删除后重新拉取：{TARGET_DIR}")
        shutil.rmtree(TARGET_DIR)

    clone_cmd = [
        "git", "clone",
        "--depth", "1",
        "--branch", REPO_BRANCH,
        REPO_URL,
        str(TARGET_DIR),
    ]
    print("  执行命令：")
    print("    " + " ".join(clone_cmd))
    _run_checked_command(clone_cmd)
    print(f"  ✓ 首次拉取完成：{TARGET_DIR}")

print("\n[2/4] 清理当前 outputs...")
outputs_dir = TARGET_DIR / "outputs"
if outputs_dir.exists():
    shutil.rmtree(outputs_dir)
    print(f"  ✓ 已删除旧输出目录：{outputs_dir}")
else:
    print("  ✓ 未发现旧 outputs 目录")
outputs_dir.mkdir(parents=True, exist_ok=True)
print(f"  ✓ 已重建 outputs 目录：{outputs_dir}")

print("\n[3/4] 校验关键目录结构...")
required_paths = [
    TARGET_DIR / "main" / "cli",
    TARGET_DIR / "configs",
    TARGET_DIR / "scripts",
    TARGET_DIR / "requirements.txt",
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"同步完成但目录结构不完整，缺失：{missing_paths}")
print("  ✓ 关键目录结构校验通过")

print("\n[4/4] 记录 Git 版本信息...")
GIT_INFO_PATH = WORK_DIR / "git_version_info.json"

try:
    commit_hash = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "rev-parse", "HEAD"
    ]).stdout.strip()
    short_hash = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "rev-parse", "--short=8", "HEAD"
    ]).stdout.strip()
    commit_message = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "log", "-1", "--pretty=%s"
    ]).stdout.strip()
    commit_timestamp = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "log", "-1", "--pretty=%ci"
    ]).stdout.strip()
    author = _run_checked_command([
        "git", "-C", str(TARGET_DIR), "log", "-1", "--pretty=%an <%ae>"
    ]).stdout.strip()

    version_info = {
        "repo_url": REPO_URL,
        "branch": REPO_BRANCH,
        "target_dir": str(TARGET_DIR),
        "sync_mode": sync_mode,
        "outputs_cleaned": True,
        "commit_hash": commit_hash,
        "commit_hash_short": short_hash,
        "commit_message": commit_message,
        "commit_timestamp": commit_timestamp,
        "commit_author": author,
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    with open(GIT_INFO_PATH, "w", encoding="utf-8") as f:
        json.dump(version_info, f, indent=2, ensure_ascii=False)

    print(f"✓ Git 版本信息已记录：{GIT_INFO_PATH}")
    print(f"  Sync mode: {sync_mode}")
    print(f"  Commit: {short_hash} - {commit_message}")
    print(f"  Author: {author}")
    print(f"  时间戳: {commit_timestamp}")
except Exception as exc:
    print(f"⚠ 版本信息记录失败（不影响主流程）：{exc}")
    fallback_info = {
        "repo_url": REPO_URL,
        "branch": REPO_BRANCH,
        "target_dir": str(TARGET_DIR),
        "sync_mode": sync_mode,
        "outputs_cleaned": True,
        "commit_hash": "<获取失败>",
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error": str(exc),
    }
    with open(GIT_INFO_PATH, "w", encoding="utf-8") as f:
        json.dump(fallback_info, f, indent=2, ensure_ascii=False)
    print(f"⚠ 已写入备用 Git 信息文件：{GIT_INFO_PATH}")

In [ ]:

# 验证和定位 CEG-WM 代码库
from pathlib import Path

def find_ceg_wm_root(base_dir):
    """
    功能：查找 CEG-WM 根目录（智能路径发现）。
    """
    base_dir = Path(base_dir)

    # 检查：特征目录结构
    characteristic_dirs = ["main/cli", "configs", "scripts", "requirements.txt"]

    # 首先检查直接目录
    if all((base_dir / d).exists() for d in characteristic_dirs):
        return base_dir

    # 查找包含正确结构的子目录
    for subdir in sorted(base_dir.iterdir()):
        if subdir.is_dir() and not subdir.name.startswith('.'):
            if all((subdir / d).exists() for d in characteristic_dirs):
                return subdir

    # 尝试找任何包含 main/cli 的目录（最后的备选方案）
    for possible_root in base_dir.rglob("main"):
        if possible_root.is_dir():
            ceg_root = possible_root.parent
            if (ceg_root / "configs").exists() and (ceg_root / "scripts").exists():
                return ceg_root

    raise FileNotFoundError(
        f"无法找到 CEG-WM 根目录\n"
        f"  搜索路径：{base_dir}\n"
        f"  期望目录结构：main/cli/, configs/, scripts/, requirements.txt\n"
        f"  请检查 GitHub 拉取步骤是否执行成功"
    )

# 定位 CEG-WM 根目录（GitHub 拉取后）
try:
    CEG_WM_ROOT = find_ceg_wm_root(WORK_DIR)
    print(f"✓ 找到 CEG-WM 根目录：{CEG_WM_ROOT}")
    print(f"  绝对路径：{CEG_WM_ROOT.resolve()}")
except FileNotFoundError as e:
    print(f"✗ {e}")
    # 列出实际的目录结构以帮助调试
    print("\n实际目录结构：")
    for item in WORK_DIR.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            print(f"  {item.name}/")
            for subitem in item.iterdir():
                if subitem.is_dir():
                    print(f"    {subitem.name}/")


In [ ]:

# 添加 CEG-WM 到 Python 路径
if str(CEG_WM_ROOT) not in sys.path:
    sys.path.insert(0, str(CEG_WM_ROOT))

# 验证关键模块可导入
print("验证关键模块导入...")
try:
    from main.cli import run_embed
    print("  ✓ main.cli.run_embed 导入成功")
except ImportError as e:
    print(f"  ✗ 导入失败：{e}")
    print("    这表示 CEG-WM 依赖未完整安装，将在下一步修复")


## 第 2 步：安装依赖包

安装 CEG-WM 项目本身和所有依赖。这是关键步骤！

In [ ]:

import subprocess
import sys

print("=" * 80)
print("安装依赖包")
print("=" * 80)

# 步骤 1：升级 pip
print("\n[1/4] 升级 pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
               capture_output=True)
print("  ✓ pip 已升级")

# 步骤 2：安装系统依赖
if IN_COLAB:
    print("\n[2/4] 安装系统依赖...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
                   capture_output=True)
    print("  ✓ 系统依赖已安装")

# 步骤 3：安装 CEG-WM 项目本身（关键！）
print("\n[3/4] 安装 CEG-WM 项目...")
try:
    # 方式 A：从 pyproject.toml（推荐）
    pyproject_file = CEG_WM_ROOT / "pyproject.toml"
    requirements_file = CEG_WM_ROOT / "requirements.txt"

    if pyproject_file.exists():
        print(f"  从 pyproject.toml 安装：{pyproject_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(CEG_WM_ROOT)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 项目安装成功（开发模式）")
        else:
            print(f"  ⚠ 项目安装失败，尝试备选方案")
            print(f"    错误：{result.stderr[-200:]}")

    elif requirements_file.exists():
        print(f"  从 requirements.txt 安装：{requirements_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 依赖安装成功")
        else:
            print(f"  ⚠ 部分依赖安装失败，继续使用...")
    else:
        print("  ⚠ 未找到 pyproject.toml 或 requirements.txt")

except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（300 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

# 步骤 4：安装 transparent-background（InSPyReNet 语义掩码模型驱动包）
print("\n[4/4] 安装 transparent-background（InSPyReNet 依赖）...")
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transparent-background"],
        capture_output=True,
        text=True,
        timeout=120
    )
    if result.returncode == 0:
        print("  ✓ transparent-background 安装成功")
    else:
        print(f"  ⚠ transparent-background 安装失败：{result.stderr[-200:]}")
        print("    InSPyReNet 语义掩码将以 fallback 模式运行（影响内容感知精度）")
except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（120 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

print("\n✓ 依赖安装步骤完成")


## 第 3 步：下载模型权重

真实下载 Stable Diffusion 3 模型。这是关键步骤！

In [ ]:
import time
import os
import gc
import torch
from pathlib import Path
from huggingface_hub import scan_cache_dir, snapshot_download

print("=" * 80)
print("下载模型权重")
print("=" * 80)

# 配置模型缓存目录
MODEL_CACHE_DIR = WORK_DIR / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

print(f"\n模型缓存目录：{MODEL_CACHE_DIR}")

print("\n检查 Hugging Face 认证...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        user_info = api.whoami()
        print(f"  ✓ 已认证：{user_info.get('name', 'Unknown')}")
    except Exception:
        print("  ℹ 未认证（使用匿名访问，私有/受限模型可能无法下载）")
except ImportError:
    print("  ⚠ huggingface_hub 未安装")

print("\n" + "=" * 80)
print("下载 Stable Diffusion 3.5 模型")
print("=" * 80)

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
print(f"目标模型：{MODEL_ID}")

def check_model_cached(model_id):
    """
    功能：检查模型是否已在缓存中。

    Check if model is already cached locally.

    Args:
        model_id: Model identifier.

    Returns:
        Boolean indicating if model is cached.
    """
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if model_id in repo.repo_id:
                return True
        return False
    except Exception:
        return False

if check_model_cached(MODEL_ID):
    print(f"✓ 模型已缓存：{MODEL_ID}")
    print("  跳过下载")
else:
    print(f"\n⏳ 模型未缓存，开始下载：{MODEL_ID}")
    print("   这可能需要 5-20 分钟（取决于网络与镜像）")

    try:
        snapshot_path = snapshot_download(
            repo_id=MODEL_ID,
            cache_dir=str(MODEL_CACHE_DIR),
            local_files_only=False,
        )
        print("  ✓ 模型下载完成")
        print(f"  本地快照路径：{snapshot_path}")
    except Exception as e:
        error_msg = str(e)
        print(f"  ✗ 模型下载失败：{e}")

        if "404" in error_msg or "Entry Not Found" in error_msg:
            print("\n  ❌ 错误：无法访问模型（404）")
            print("  原因：SD3.5 可能需要先在 Hugging Face 页面授权")
        elif "401" in error_msg or "Unauthorized" in error_msg or "403" in error_msg:
            print("\n  ❌ 错误：认证失败（401/403）")
            print("  原因：HF_TOKEN 无效、缺失或未获模型访问权限")
        else:
            print("\n  解决方案：")
            print("    1. 检查网络连接")
            print("    2. 检查 HF_TOKEN 与模型访问授权")
            print("    3. 重新执行本 cell")

        print("\n  ⚠️ 继续执行（后续步骤可能失败）...")

print("\n清理内存...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("  ✓ GPU 缓存已清理")

print("✓ 模型准备完成")

In [ ]:
import os
import shutil
import urllib.request
from pathlib import Path

print("=" * 80)
print("下载 InSPyReNet 权重（paper_full_cuda）")
print("=" * 80)

if 'WORK_DIR' not in globals():
    raise RuntimeError("请先执行前序单元，确保 WORK_DIR 已初始化")

if 'CEG_WM_ROOT' in globals():
    repo_inspyrenet_model_path = Path(CEG_WM_ROOT) / "models" / "inspyrenet" / "ckpt_base.pth"
else:
    repo_inspyrenet_model_path = None

# Colab 同步仓库时不包含未跟踪的 models 目录，因此把可复用权重放在 WORK_DIR 下的持久路径。
cached_inspyrenet_model_path = Path(WORK_DIR) / "models" / "inspyrenet" / "ckpt_base.pth"
inspyrenet_model_path = repo_inspyrenet_model_path or cached_inspyrenet_model_path

cached_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)
if repo_inspyrenet_model_path is not None:
    repo_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)

inspyrenet_repo_tree_url = "https://huggingface.co/plemeri/InSPyReNet/tree/main"
default_inspyrenet_url = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"

inspyrenet_model_url = os.environ.get(
    "INSPYRENET_MODEL_URL",
    default_inspyrenet_url,
).strip()

force_inspyrenet_download = os.environ.get(
    "INSPYRENET_FORCE_DOWNLOAD",
    "",
).strip().lower() in {"1", "true", "yes"}


def _is_valid_weight_file(file_path) -> bool:
    if file_path is None:
        return False
    return file_path.exists() and file_path.is_file() and file_path.stat().st_size > 0


def _sync_weight_to_repo(source_path: Path, target_path) -> None:
    if target_path is None:
        return
    if source_path.resolve() == target_path.resolve():
        return
    shutil.copy2(source_path, target_path)


print(f"workflow 目标权重路径：{inspyrenet_model_path}")
print(f"持久缓存路径：{cached_inspyrenet_model_path}")
print(f"仓库页面（核对用）：{inspyrenet_repo_tree_url}")
print(f"下载地址（程序使用）：{inspyrenet_model_url}")
print(f"强制重下：{force_inspyrenet_download}")

existing_weight_path = None
candidate_paths = []
if repo_inspyrenet_model_path is not None:
    candidate_paths.append(repo_inspyrenet_model_path)
candidate_paths.append(cached_inspyrenet_model_path)

for candidate_path in candidate_paths:
    if _is_valid_weight_file(candidate_path):
        existing_weight_path = candidate_path
        break

download_required = force_inspyrenet_download or existing_weight_path is None
if existing_weight_path is not None and not force_inspyrenet_download:
    existing_size = existing_weight_path.stat().st_size
    print(f"✓ 已发现现有 InSPyReNet 权重：{existing_weight_path}（size={existing_size} bytes）")
    _sync_weight_to_repo(existing_weight_path, repo_inspyrenet_model_path)
    if existing_weight_path != cached_inspyrenet_model_path:
        shutil.copy2(existing_weight_path, cached_inspyrenet_model_path)
    print("✓ 已复用现有权重，不执行重复下载")

if download_required:
    temp_download_path = cached_inspyrenet_model_path.with_suffix(cached_inspyrenet_model_path.suffix + ".downloading")
    if temp_download_path.exists():
        temp_download_path.unlink()

    print(f"开始下载：{inspyrenet_model_url}")
    urllib.request.urlretrieve(inspyrenet_model_url, str(temp_download_path))

    if not _is_valid_weight_file(temp_download_path):
        temp_download_path.unlink(missing_ok=True)
        raise RuntimeError(f"下载完成但权重文件无效：{temp_download_path}")

    temp_download_path.replace(cached_inspyrenet_model_path)
    _sync_weight_to_repo(cached_inspyrenet_model_path, repo_inspyrenet_model_path)
    print("✓ 下载完成")

if _is_valid_weight_file(repo_inspyrenet_model_path):
    final_weight_path = repo_inspyrenet_model_path
else:
    final_weight_path = cached_inspyrenet_model_path

final_weight_size = final_weight_path.stat().st_size if _is_valid_weight_file(final_weight_path) else 0

print(f"最终 workflow 使用路径：{final_weight_path}")
print(f"最终权重大小：{final_weight_size} bytes")
print("✓ InSPyReNet 准备完成")

## 第 4 步：配置选择、数据准备与 attestation 环境变量

选择工作流配置并准备输入数据，然后设置 formal GPU 验收必需的 attestation 密钥环境变量。

In [ ]:
import json

print("=" * 80)
print("工作流配置和数据准备")
print("=" * 80)

# 选择配置文件。
# 注意：此处 CONFIG_CHOICE 固定为 "paper_full_cuda"，与第 5 步主流程完全对齐。
# 第 5 步不依赖此变量选择 profile，但会检查它是否已定义以确认前序 cell 已执行。
CONFIG_CHOICE = "paper_full_cuda"
print(f"\n选定配置：{CONFIG_CHOICE}")

CONFIG_FILE = CEG_WM_ROOT / "configs" / f"{CONFIG_CHOICE}.yaml"
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"paper_full_cuda 配置不存在：{CONFIG_FILE}")

PAPER_SPEC_FILE = CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml"
if PAPER_SPEC_FILE.exists():
    print(f"  ✓ 论文一致性规范存在：{PAPER_SPEC_FILE.name}")
else:
    print(f"  ⚠ 未找到论文一致性规范：{PAPER_SPEC_FILE}")

MODEL_CFG_FILE = CEG_WM_ROOT / "configs" / "model_sd3.yaml"
if MODEL_CFG_FILE.exists():
    print(f"  ✓ SD3 模型配置存在：{MODEL_CFG_FILE.name}")
else:
    print(f"  ⚠ 未找到 SD3 模型配置：{MODEL_CFG_FILE}")

print(f"  配置文件路径：{CONFIG_FILE}")
print(f"  配置文件大小：{CONFIG_FILE.stat().st_size / 1024:.1f} KB")

# 创建数据目录
data_dir = CEG_WM_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 准备输入数据
print("\n准备输入数据...")


In [ ]:
import json
import os
import re
import secrets
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("准备 attestation 环境变量（formal GPU 验收必需）")
print("=" * 80)

ATTESTATION_ENV_SPECS = {
    "CEG_WM_K_MASTER": 64,
    "CEG_WM_K_PROMPT": 32,
    "CEG_WM_K_SEED": 32,
}

# 如需固定密钥，请把对应值填成十六进制字符串；留空则优先复用现有环境变量。
USER_ATTESTATION_VALUES = {
    "CEG_WM_K_MASTER": "",
    "CEG_WM_K_PROMPT": "",
    "CEG_WM_K_SEED": "",
}

# 为了让 Colab 首次执行可直接通过 preflight，默认会为缺失项生成本次会话内有效的临时密钥。
AUTO_GENERATE_MISSING_ATTESTATION_KEYS = True

ATTESTATION_INFO_PATH = WORK_DIR / "attestation_env_keys.json"

def _is_valid_hex_secret(value: str, expected_length: int) -> bool:
    if not isinstance(value, str):
        return False
    candidate = value.strip().lower()
    if len(candidate) != expected_length:
        return False
    return re.fullmatch(r"[0-9a-f]+", candidate) is not None

def _mask_secret(value: str) -> str:
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"

resolved_values = {}
generated_env_vars = []
invalid_env_vars = []

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    manual_value = USER_ATTESTATION_VALUES.get(env_name, "")
    existing_value = os.environ.get(env_name, "")

    if _is_valid_hex_secret(manual_value, expected_length):
        resolved_value = manual_value.strip().lower()
        source = "manual_input"
    elif _is_valid_hex_secret(existing_value, expected_length):
        resolved_value = existing_value.strip().lower()
        source = "existing_env"
    elif AUTO_GENERATE_MISSING_ATTESTATION_KEYS:
        resolved_value = secrets.token_hex(expected_length // 2)
        source = "generated_for_session"
        generated_env_vars.append(env_name)
    else:
        resolved_value = ""
        source = "missing"

    if resolved_value:
        os.environ[env_name] = resolved_value
        resolved_values[env_name] = {
            "length": len(resolved_value),
            "masked_value": _mask_secret(resolved_value),
            "value": resolved_value,
            "source": source,
        }
    else:
        invalid_env_vars.append(env_name)

print("\nattestation 环境变量状态：")
for env_name in ATTESTATION_ENV_SPECS:
    if env_name in resolved_values:
        item = resolved_values[env_name]
        print(
            f"  ✓ {env_name}: source={item['source']}, len={item['length']}, value={item['masked_value']}"
        )
    else:
        print(f"  ✗ {env_name}: 缺失或格式非法")

if generated_env_vars:
    print("\nℹ 已为当前 Notebook 会话自动生成临时 attestation 密钥：")
    for env_name in generated_env_vars:
        print(f"  - {env_name}")
    print("  重新启动内核后需要再次执行本 cell。")

if invalid_env_vars:
    raise RuntimeError(
        "以下 attestation 环境变量仍未就绪，请填写合法十六进制值或保持自动生成开启："
        f" {invalid_env_vars}"
    )

attestation_info = {
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "work_dir": str(WORK_DIR),
    "auto_generate_missing_attestation_keys": bool(AUTO_GENERATE_MISSING_ATTESTATION_KEYS),
    "generated_env_vars": generated_env_vars,
    "env_specs": ATTESTATION_ENV_SPECS,
    "resolved_values": resolved_values,
}

with open(ATTESTATION_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(attestation_info, f, indent=2, ensure_ascii=False)

print(f"\n✓ attestation 环境变量与密钥已保存：{ATTESTATION_INFO_PATH}")
print("✓ attestation 环境变量准备完成，可继续执行运行前预检与第 5 步")

In [ ]:
# 第 4.5 步：paper_full_cuda 运行前预检（建议在第 5 步前执行）
import os
import shutil
import socket
from pathlib import Path

print("=" * 80)
print("运行前预检：paper_full_cuda")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals():
    raise RuntimeError("请先执行前面的单元，确保 CEG_WM_ROOT 已初始化")

precheck_results = []

def _record_check(name: str, ok: bool, detail: str):
    precheck_results.append({"name": name, "ok": ok, "detail": detail})
    status = "✓" if ok else "✗"
    print(f"{status} {name}: {detail}")

# 1) CUDA / GPU 预检
try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        _record_check("CUDA 可用", True, f"device={gpu_name}")
    else:
        _record_check("CUDA 可用", False, "未检测到 GPU，请在 Colab 切换到 GPU Runtime")
except Exception as exc:
    _record_check("CUDA 可用", False, f"torch 检查异常: {type(exc).__name__}: {exc}")

# 2) 关键配置与脚本存在性预检
required_paths = [
    CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml",
    CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml",
    CEG_WM_ROOT / "scripts" / "run_onefile_workflow.py",
]
for path in required_paths:
    _record_check(f"文件存在: {path.name}", path.exists(), str(path))

# 2.2) attestation 环境变量预检（formal 路径为硬性要求）
attestation_env_specs = {
    "CEG_WM_K_MASTER": 64,
    "CEG_WM_K_PROMPT": 32,
    "CEG_WM_K_SEED": 32,
}
for env_name, expected_length in attestation_env_specs.items():
    value = os.environ.get(env_name, "").strip().lower()
    is_hex = len(value) == expected_length and all(ch in "0123456789abcdef" for ch in value)
    detail = f"len={len(value)}" if value else "<absent>"
    _record_check(f"attestation 环境变量: {env_name}", is_hex, detail)

# 2.5) InSPyReNet 权重文件预检（与 paper_full_cuda 相对路径语义对齐）
inspyrenet_weight_path = CEG_WM_ROOT / "models" / "inspyrenet" / "ckpt_base.pth"
if inspyrenet_weight_path.exists() and inspyrenet_weight_path.is_file():
    try:
        weight_size = inspyrenet_weight_path.stat().st_size
        inspyrenet_ok = weight_size > 0
        _record_check(
            "InSPyReNet 权重存在",
            inspyrenet_ok,
            f"path={inspyrenet_weight_path}, size={weight_size} bytes",
        )
    except Exception as exc:
        _record_check(
            "InSPyReNet 权重存在",
            False,
            f"读取文件信息失败: {type(exc).__name__}: {exc}",
        )
else:
    _record_check(
        "InSPyReNet 权重存在",
        False,
        f"未找到权重文件: {inspyrenet_weight_path}（请先执行第 3 步 InSPyReNet 下载单元）",
    )

# 3) Python 包可用性预检
required_modules = ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors"]
for module_name in required_modules:
    try:
        __import__(module_name)
        _record_check(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        _record_check(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

# 4) Hugging Face 网络与访问预检（非阻塞，但强烈建议通过）
hf_ok = False
hf_detail = "未执行"
try:
    from huggingface_hub import HfApi
    socket.setdefaulttimeout(15)
    api = HfApi()
    _ = api.model_info("stabilityai/stable-diffusion-3.5-medium")
    hf_ok = True
    hf_detail = "可访问 stabilityai/stable-diffusion-3.5-medium"
except Exception as exc:
    hf_ok = False
    hf_detail = f"访问失败（可能是网络/权限问题）: {type(exc).__name__}: {exc}"
_record_check("HF 模型可访问", hf_ok, hf_detail)

# 5) 工作空间磁盘空间预检
usage = shutil.disk_usage(str(CEG_WM_ROOT))
free_gb = usage.free / (1024 ** 3)
disk_ok = free_gb >= 20.0
_record_check("磁盘剩余空间", disk_ok, f"free={free_gb:.1f} GB，建议 >= 20 GB")

# 汇总
hard_fail_names = {
    "CUDA 可用",
    "文件存在: paper_full_cuda.yaml",
    "文件存在: run_onefile_workflow.py",
    "attestation 环境变量: CEG_WM_K_MASTER",
    "attestation 环境变量: CEG_WM_K_PROMPT",
    "attestation 环境变量: CEG_WM_K_SEED",
    "InSPyReNet 权重存在",
    "模块可导入: diffusers",
    "模块可导入: transformers",
}
hard_fail = [item for item in precheck_results if (item["name"] in hard_fail_names and not item["ok"])]
soft_fail = [item for item in precheck_results if (item["name"] not in hard_fail_names and not item["ok"])]

print("\n" + "-" * 80)
print("预检结果汇总")
print("-" * 80)
print(f"硬性失败项数量: {len(hard_fail)}")
print(f"软性失败项数量: {len(soft_fail)}")

if hard_fail:
    print("\n✗ 预检未通过（存在硬性失败项），请先修复后再执行第 5 步。")
    for item in hard_fail:
        print(f"  - {item['name']}: {item['detail']}")
else:
    if soft_fail:
        print("\n✓ 预检通过（存在软性风险项，建议优先处理）：")
        for item in soft_fail:
            print(f"  - {item['name']}: {item['detail']}")
    else:
        print("\n✓ 预检全部通过，可以执行第 5 步。")

In [ ]:
# 第 4.8 步：S2 attention layer 命名真实探针（SD3.5）+ 打包下载（前置执行）
import json
import zipfile
import traceback
from datetime import datetime
from pathlib import Path

import torch

print("=" * 80)
print("S2 探针：SD3.5 self-attention layer 命名与运行时命中验证（workflow 前执行）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals():
    raise RuntimeError("请先执行前序单元，确保 CEG_WM_ROOT 已初始化")

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA。请在 Colab 切换到 GPU Runtime 后再执行本单元。")

probe_root = Path(CEG_WM_ROOT) / "outputs" / "s2_attention_probe"
probe_root.mkdir(parents=True, exist_ok=True)

print(f"输出目录：{probe_root}")

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
PROMPT = "a clean geometric abstract pattern"
NEG_PROMPT = "blurry, noisy"
HEIGHT = 256
WIDTH = 256
NUM_STEPS = 2
GUIDANCE_SCALE = 3.5
MAX_LAYERS = 8
SEED = 42

result = {
    "probe_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model_id": MODEL_ID,
    "device": "cuda",
    "config": {
        "height": HEIGHT,
        "width": WIDTH,
        "num_inference_steps": NUM_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "max_layers": MAX_LAYERS,
        "seed": SEED,
    },
    "status": "started",
    "errors": [],
    "static_candidates": [],
    "runtime_hits": [],
    "paired_layers": [],
    "summary": {},
}

script_path = probe_root / "s2_probe_script.py"
script_source = r'''
import torch
from diffusers import StableDiffusion3Pipeline

class QKCaptureHook:
    def __init__(self, max_layers=None):
        self.max_layers = max_layers
        self.handles = []
        self.hits = {}
        self.step_index = 0

    def _make_hook(self, module_name, projection):
        def hook_fn(module, inputs, output):
            key = (module_name, int(self.step_index))
            if key not in self.hits:
                self.hits[key] = {}
            shape = tuple(output.shape) if hasattr(output, "shape") else None
            self.hits[key][projection] = {"shape": shape}
        return hook_fn

    def register(self, transformer):
        hooked_layers = 0
        for name, module in transformer.named_modules():
            if ".attn2." in name:
                continue
            if self.max_layers is not None and hooked_layers >= int(self.max_layers):
                break
            if name.endswith(".to_q"):
                parent = name.rsplit(".", 1)[0]
                self.handles.append(module.register_forward_hook(self._make_hook(parent, "q")))
            elif name.endswith(".to_k"):
                parent = name.rsplit(".", 1)[0]
                self.handles.append(module.register_forward_hook(self._make_hook(parent, "k")))
                hooked_layers += 1

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []
'''
with open(script_path, "w", encoding="utf-8") as f:
    f.write(script_source)

try:
    from diffusers import StableDiffusion3Pipeline
except Exception as exc:
    raise RuntimeError(f"diffusers 导入失败：{type(exc).__name__}: {exc}")

class QKCaptureHook:
    """
    功能：捕获 SD3.5 Transformer 中 self-attention 的 Q/K 投影命中。
    """
    def __init__(self, max_layers=None):
        self.max_layers = max_layers
        self.handles = []
        self.hits = {}
        self.step_index = 0

    def _make_hook(self, module_name, projection):
        def hook_fn(module, inputs, output):
            key = (module_name, int(self.step_index))
            if key not in self.hits:
                self.hits[key] = {}
            output_shape = tuple(output.shape) if hasattr(output, "shape") else None
            self.hits[key][projection] = {
                "shape": output_shape,
            }
        return hook_fn

    def register(self, transformer):
        hooked_layers = 0
        for name, module in transformer.named_modules():
            if ".attn2." in name:
                continue
            if self.max_layers is not None and hooked_layers >= int(self.max_layers):
                break
            if name.endswith(".to_q"):
                parent = name.rsplit(".", 1)[0]
                self.handles.append(module.register_forward_hook(self._make_hook(parent, "q")))
            elif name.endswith(".to_k"):
                parent = name.rsplit(".", 1)[0]
                self.handles.append(module.register_forward_hook(self._make_hook(parent, "k")))
                hooked_layers += 1

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

def _collect_pairs(hits):
    runtime_hits = []
    paired_layers = {}
    for (module_name, step_idx), payload in sorted(hits.items(), key=lambda x: (x[0][1], x[0][0])):
        q_shape = payload.get("q", {}).get("shape") if isinstance(payload.get("q"), dict) else None
        k_shape = payload.get("k", {}).get("shape") if isinstance(payload.get("k"), dict) else None
        runtime_hits.append({
            "module_name": module_name,
            "step_index": step_idx,
            "has_q": "q" in payload,
            "has_k": "k" in payload,
            "q_shape": q_shape,
            "k_shape": k_shape,
        })

        if module_name not in paired_layers:
            paired_layers[module_name] = {"q": 0, "k": 0, "q_shape": None, "k_shape": None}
        if "q" in payload:
            paired_layers[module_name]["q"] += 1
            paired_layers[module_name]["q_shape"] = q_shape
        if "k" in payload:
            paired_layers[module_name]["k"] += 1
            paired_layers[module_name]["k_shape"] = k_shape

    paired_summary = []
    for module_name, stats in sorted(paired_layers.items(), key=lambda x: x[0]):
        paired_summary.append({
            "module_name": module_name,
            "q_hits": stats["q"],
            "k_hits": stats["k"],
            "paired_ok": bool(stats["q"] > 0 and stats["k"] > 0),
            "q_shape": stats["q_shape"],
            "k_shape": stats["k_shape"],
        })
    return runtime_hits, paired_summary

def _run_probe_once(height, width, steps, max_layers):
    torch.cuda.empty_cache()
    pipe = StableDiffusion3Pipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
    pipe = pipe.to("cuda")
    pipe.set_progress_bar_config(disable=True)
    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()
    if hasattr(pipe, "vae") and pipe.vae is not None and hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()

    transformer = getattr(pipe, "transformer", None)
    if transformer is None:
        raise RuntimeError("pipeline.transformer is absent")

    static_candidates = [
        name
        for name, _ in transformer.named_modules()
        if name.endswith(".to_q") or name.endswith(".to_k")
    ]

    hook = QKCaptureHook(max_layers=max_layers)
    try:
        hook.register(transformer)
        generator = torch.Generator(device="cuda").manual_seed(SEED)

        def _on_step_end(pipeline, step_index, timestep, callback_kwargs):
            hook.step_index = int(step_index)
            return callback_kwargs

        pipe_kwargs = {
            "prompt": PROMPT,
            "negative_prompt": NEG_PROMPT,
            "num_inference_steps": steps,
            "guidance_scale": GUIDANCE_SCALE,
            "height": height,
            "width": width,
            "generator": generator,
            "output_type": "pil",
        }

        try:
            _ = pipe(**pipe_kwargs, callback_on_step_end=_on_step_end)
        except TypeError:
            _ = pipe(**pipe_kwargs)
    finally:
        hook.remove()

    runtime_hits, paired_summary = _collect_pairs(hook.hits)

    del pipe
    torch.cuda.empty_cache()

    return static_candidates, runtime_hits, paired_summary

print("\n[1/5] 运行 attention probe（先标准配置，再失败降级）...")
probe_attempts = [
    {"height": HEIGHT, "width": WIDTH, "steps": NUM_STEPS, "max_layers": MAX_LAYERS, "tag": "primary"},
    {"height": 256, "width": 256, "steps": 1, "max_layers": 4, "tag": "fallback_low_mem"},
]

probe_ok = False
for attempt in probe_attempts:
    print(
        f"  -> 尝试 {attempt['tag']}: "
        f"{attempt['height']}x{attempt['width']}, steps={attempt['steps']}, max_layers={attempt['max_layers']}"
    )
    try:
        static_candidates, runtime_hits, paired_summary = _run_probe_once(
            height=attempt["height"],
            width=attempt["width"],
            steps=attempt["steps"],
            max_layers=attempt["max_layers"],
        )
        result["config_used"] = attempt
        result["static_candidates"] = static_candidates
        result["runtime_hits"] = runtime_hits
        result["paired_layers"] = paired_summary
        probe_ok = True
        print("  ✓ 探针执行成功")
        break
    except Exception as exc:
        err = f"{attempt['tag']} failed: {type(exc).__name__}: {exc}"
        result["errors"].append(err)
        result["errors"].append(traceback.format_exc())
        print(f"  ✗ {err}")

if not probe_ok:
    result["status"] = "failed"
else:
    paired_ok_count = sum(1 for item in result.get("paired_layers", []) if item.get("paired_ok"))
    result["summary"] = {
        "static_candidate_count": len(result.get("static_candidates", [])),
        "runtime_hit_items": len(result.get("runtime_hits", [])),
        "paired_layer_count": len(result.get("paired_layers", [])),
        "paired_ok_count": int(paired_ok_count),
    }
    result["status"] = "ok"

print("\n[2/5] 写出结果文件...")
json_path = probe_root / "s2_attention_probe_result.json"
txt_path = probe_root / "s2_attention_probe_summary.txt"
err_path = probe_root / "s2_attention_probe_errors.log"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

summary_lines = [
    "=" * 80,
    "S2 SD3.5 Attention Probe Summary",
    "=" * 80,
    f"status: {result.get('status')}",
    f"model_id: {result.get('model_id')}",
    f"config_used: {result.get('config_used', {})}",
    f"static_candidate_count: {result.get('summary', {}).get('static_candidate_count')}",
    f"runtime_hit_items: {result.get('summary', {}).get('runtime_hit_items')}",
    f"paired_layer_count: {result.get('summary', {}).get('paired_layer_count')}",
    f"paired_ok_count: {result.get('summary', {}).get('paired_ok_count')}",
    "",
    "Top paired layers:",
]

for item in result.get("paired_layers", [])[:20]:
    summary_lines.append(
        f"- {item.get('module_name')} | paired_ok={item.get('paired_ok')} | "
        f"q_hits={item.get('q_hits')} | k_hits={item.get('k_hits')} | "
        f"q_shape={item.get('q_shape')} | k_shape={item.get('k_shape')}"
    )

if result.get("errors"):
    summary_lines.append("")
    summary_lines.append("Errors:")
    for err in result.get("errors", [])[:10]:
        summary_lines.append(f"- {err}")

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

with open(err_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(result.get("errors", [])))

print("\n[3/5] 打包下载文件...")
zip_path = probe_root.parent / "s2_attention_probe_bundle.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_path, arcname="s2_attention_probe/s2_attention_probe_result.json")
    zf.write(txt_path, arcname="s2_attention_probe/s2_attention_probe_summary.txt")
    zf.write(err_path, arcname="s2_attention_probe/s2_attention_probe_errors.log")
    zf.write(script_path, arcname="s2_attention_probe/s2_probe_script.py")

S2_PROBE_ROOT = probe_root
S2_PROBE_BUNDLE_PATH = zip_path

print(f"  ✓ 结果 JSON：{json_path}")
print(f"  ✓ 摘要 TXT：{txt_path}")
print(f"  ✓ 错误日志：{err_path}")
print(f"  ✓ 打包 ZIP：{zip_path}")

print("\n[4/5] 触发 Colab 下载...")
if 'IN_COLAB' in globals() and IN_COLAB:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
        print(f"  ✓ 已触发下载：{zip_path.name}")
    except Exception as exc:
        print(f"  ⚠ 自动下载失败，请手动下载：{zip_path}，原因：{exc}")
else:
    print(f"  ℹ 非 Colab 环境，请手动取用：{zip_path}")

print("\n[5/5] 完成")
print("请把下载的 s2_attention_probe_bundle.zip 发给我，我会基于真实结果给出 S2 最终层名与 hook 白名单。")

## 第 5 步：formal GPU 验收 workflow 执行

本步仅负责执行正式 workflow 主链，并写出 workflow 日志、formal summary 及 run_root 下的阶段产物。

**执行前提**：
- 已完成前面的环境准备、依赖安装、模型权重准备与 attestation 环境变量设置。
- 已完成 workflow 前置的 S2 attention probe。

**执行策略**：
- 强制要求 CUDA 可用。
- 强制使用真实模型 stable-diffusion-3.5-medium。
- 主流程调用 scripts/run_paper_full_workflow_verification.py，避免在 Notebook 中复制正式主链。
- 额外产出 artifacts/workflow_acceptance/paper_full_formal_summary.json，用于区分环境阻断与代码失败，并固化 attestation / freeze-gate 相关状态。

**执行后分支**：
- 如果本步完成，下一步先执行第 5.1 步，优先检查并导出完整项目结果包。
- 如果第 5.1 步提示缺失必需产物并禁止打包，则继续执行第 6 步验证输出，再执行第 7 步生成失败诊断与审计包。

In [ ]:
import json
import sys
import subprocess
import shutil
from pathlib import Path
import torch

print("=" * 80)
print("第 5 步：formal GPU 验收 workflow")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'CONFIG_CHOICE' not in globals():
    raise RuntimeError("请先执行前面的环境与配置 cell（第 1-4 步）")

if not torch.cuda.is_available():
    raise RuntimeError("当前运行时未检测到 CUDA。请在 Colab 切换到 GPU Runtime 后重试。")

gpu_name = torch.cuda.get_device_name(0)
print(f"✓ CUDA 可用，设备：{gpu_name}")

RUN_ROOT = CEG_WM_ROOT / "outputs" / "colab_run_paper_full_cuda"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

for name in ["records", "artifacts", "logs"]:
    target = RUN_ROOT / name
    if target.exists():
        shutil.rmtree(target)

logs_dir = RUN_ROOT / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

PROJECT_PAPER_CFG = CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml"
if not PROJECT_PAPER_CFG.exists():
    raise FileNotFoundError(f"未找到项目配置：{PROJECT_PAPER_CFG}")

ACTIVE_CONFIG_FILE = PROJECT_PAPER_CFG
ACTIVE_WORKFLOW_PROFILE = "paper_full_cuda_formal"
ACTIVE_SIGNOFF_PROFILE = "paper"
WORKFLOW_SUMMARY_PATH = RUN_ROOT / "artifacts" / "workflow_acceptance" / "paper_full_formal_summary.json"

print(f"✓ 输出目录：{RUN_ROOT}")
print(f"✓ 运行配置：{ACTIVE_CONFIG_FILE}")
print(f"✓ formal summary 目标：{WORKFLOW_SUMMARY_PATH}")

command = [
    sys.executable,
    "scripts/run_paper_full_workflow_verification.py",
    "--config",
    str(ACTIVE_CONFIG_FILE),
    "--run-root",
    str(RUN_ROOT),
]

print("\n执行 formal GPU 验收脚本...")
result = subprocess.run(
    command,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

workflow_log = logs_dir / "workflow_execution.log"
workflow_stdout_full_log = logs_dir / "workflow_stdout_full.log"
workflow_stderr_full_log = logs_dir / "workflow_stderr_full.log"
workflow_command_json = logs_dir / "workflow_command.json"

with open(workflow_stdout_full_log, "w", encoding="utf-8") as f:
    f.write(result.stdout or "")

with open(workflow_stderr_full_log, "w", encoding="utf-8") as f:
    f.write(result.stderr or "")

command_meta = {
    "command": command,
    "cwd": str(CEG_WM_ROOT),
    "return_code": int(result.returncode),
    "stdout_log": str(workflow_stdout_full_log),
    "stderr_log": str(workflow_stderr_full_log),
    "summary_path": str(WORKFLOW_SUMMARY_PATH),
}
with open(workflow_command_json, "w", encoding="utf-8") as f:
    json.dump(command_meta, f, indent=2, ensure_ascii=False)

with open(workflow_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(command))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(result.stderr or "")

formal_summary = {}
if WORKFLOW_SUMMARY_PATH.exists():
    with open(WORKFLOW_SUMMARY_PATH, "r", encoding="utf-8") as f:
        formal_summary = json.load(f)

print(f"返回码：{result.returncode}")
print(f"完整 stdout 日志：{workflow_stdout_full_log}")
print(f"完整 stderr 日志：{workflow_stderr_full_log}")
print(f"命令元信息：{workflow_command_json}")
print(f"formal summary：{WORKFLOW_SUMMARY_PATH}")

if result.stdout:
    print("\nSTDOUT（最后 30 行）：")
    print("\n".join(result.stdout.splitlines()[-30:]))

if result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(result.stderr.splitlines()[-20:]))

if formal_summary:
    print("\nformal summary 摘要：")
    print(f"  profile_role: {formal_summary.get('profile_role')}")
    print(f"  workflow_exit_code: {formal_summary.get('workflow_exit_code')}")
    print(f"  environment_blocked: {formal_summary.get('environment_blocked')}")
    print(f"  formal_output_expectation_ok: {formal_summary.get('formal_output_expectation_ok')}")
    print(f"  attestation_bundle_generated: {formal_summary.get('attestation_bundle_generated')}")
    print(f"  signoff_decision: {formal_summary.get('signoff_decision')}")
    preflight = formal_summary.get("details", {}).get("preflight", {})
    if preflight:
        print(f"  preflight: {preflight}")

if result.returncode == 0:
    STAGE_STATUS = "success"
elif result.returncode == 2 and formal_summary.get("environment_blocked") is True:
    STAGE_STATUS = "environment_blocked"
else:
    STAGE_STATUS = f"failed (rc={result.returncode})"

print("\n" + "=" * 80)
if result.returncode == 0:
    print("✓ 第 5 步执行成功！")
    print(f"  结果目录：{RUN_ROOT}")
elif result.returncode == 2 and formal_summary.get("environment_blocked") is True:
    print("⚠ 第 5 步被环境前置条件阻断")
    print(f"  formal summary：{WORKFLOW_SUMMARY_PATH}")
    print("  请先修复 GPU / attestation 环境变量，再重新运行第 5 步")
else:
    print("✗ 第 5 步执行失败")
    print(f"  返回码：{result.returncode}")
    print(f"  日志文件：{workflow_log}")
    print("\n提示：请运行下一个 cell（快速诊断）查看具体问题")

print("=" * 80)

In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("第 5.1 步：优先打包 formal workflow 项目结果")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再打包项目结果")

run_root = Path(RUN_ROOT)
records_dir = run_root / "records"
artifacts_dir = run_root / "artifacts"
outputs_dir = run_root / "outputs"
logs_dir = run_root / "logs"

PROJECT_EXPORT_STATUS = "not_started"
PROJECT_EXPORT_DOWNLOADED = False
PROJECT_RESULTS_ARCHIVE_PATH = None
PROJECT_RESULTS_ARCHIVE_SYNC_PATH = None
PROJECT_EXPORT_MISSING_FILES = []
PROJECT_EXPORT_FILE_COUNT = 0


def _ensure_git_info_file() -> Path:
    existing_git_info_path = globals().get("GIT_INFO_PATH")
    if existing_git_info_path is not None:
        candidate_path = Path(existing_git_info_path)
        if candidate_path.exists() and candidate_path.is_file():
            return candidate_path

    if 'CEG_WM_ROOT' not in globals():
        raise RuntimeError("缺少 CEG_WM_ROOT，无法补生成 Git 信息")

    repo_root = Path(CEG_WM_ROOT)
    git_info_path = WORK_DIR / "git_version_info.json"

    def _run_git(command):
        result = subprocess.run(
            command,
            cwd=str(repo_root),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Git 信息补生成失败，return_code={result.returncode}，command={' '.join(command)}"
            )
        return result.stdout.strip()

    version_info = {
        "repo_url": _run_git(["git", "remote", "get-url", "origin"]),
        "branch": _run_git(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
        "target_dir": str(repo_root),
        "sync_mode": globals().get("sync_mode", "<unknown>"),
        "outputs_cleaned": True,
        "commit_hash": _run_git(["git", "rev-parse", "HEAD"]),
        "commit_hash_short": _run_git(["git", "rev-parse", "--short=8", "HEAD"]),
        "commit_message": _run_git(["git", "log", "-1", "--pretty=%s"]),
        "commit_timestamp": _run_git(["git", "log", "-1", "--pretty=%ci"]),
        "commit_author": _run_git(["git", "log", "-1", "--pretty=%an <%ae>"]),
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "generated_for_export": True,
    }

    with open(git_info_path, "w", encoding="utf-8") as f:
        json.dump(version_info, f, indent=2, ensure_ascii=False)

    globals()["GIT_INFO_PATH"] = git_info_path
    print(f"✓ 已补生成 Git 信息：{git_info_path}")
    return git_info_path


def _build_timestamped_filename(base_name: str, suffix: str) -> str:
    timestamp_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{base_name}_{timestamp_suffix}{suffix}"


def _sync_file_to_google_drive(source_path: Path) -> Path | None:
    drive_export_dir = globals().get("GOOGLE_DRIVE_EXPORT_DIR")
    if drive_export_dir is None:
        return None

    drive_export_dir = Path(drive_export_dir)
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    target_path = drive_export_dir / source_path.name
    shutil.copy2(source_path, target_path)
    return target_path


try:
    git_info_path = _ensure_git_info_file()
except Exception as exc:
    git_info_path = None
    PROJECT_EXPORT_MISSING_FILES.append(f"<Git 信息不可用: {exc}>")

required_files = list(globals().get("REQUIRED_PACKAGE_FILES", []))
if not required_files:
    required_files = [
        records_dir / "embed_record.json",
        records_dir / "detect_record.json",
        records_dir / "calibration_record.json",
        records_dir / "evaluate_record.json",
        artifacts_dir / "evaluation_report.json",
        artifacts_dir / "run_closure.json",
        artifacts_dir / "signoff" / "signoff_report.json",
        artifacts_dir / "workflow_acceptance" / "paper_full_formal_summary.json",
        artifacts_dir / "attestation" / "attestation_statement.json",
        artifacts_dir / "attestation" / "attestation_bundle.json",
        artifacts_dir / "attestation" / "attestation_result.json",
        artifacts_dir / "thresholds" / "thresholds_artifact.json",
        outputs_dir / "experiment_matrix" / "artifacts" / "grid_summary.json",
    ]
required_files = [Path(path) for path in required_files]

required_support_files = [
    logs_dir / "workflow_execution.log",
    logs_dir / "workflow_stdout_full.log",
    logs_dir / "workflow_stderr_full.log",
    logs_dir / "workflow_command.json",
]
if 'ATTESTATION_INFO_PATH' in globals():
    required_support_files.append(Path(ATTESTATION_INFO_PATH))
else:
    PROJECT_EXPORT_MISSING_FILES.append("<ATTESTATION_INFO_PATH 未定义>")
if git_info_path is not None:
    required_support_files.append(git_info_path)

missing_required_files = [str(path) for path in required_files if not path.exists()]
missing_support_files = [str(path) for path in required_support_files if not Path(path).exists()]
PROJECT_EXPORT_MISSING_FILES.extend(missing_required_files)
PROJECT_EXPORT_MISSING_FILES.extend(missing_support_files)

if PROJECT_EXPORT_MISSING_FILES:
    PROJECT_EXPORT_STATUS = "missing_required_files"
    print("✗ 检测到缺失必需产物，当前禁止打包：")
    for item in PROJECT_EXPORT_MISSING_FILES:
        print(f"  - {item}")
    print("请先执行第 6 步验证工作流输出，再执行第 7 步快速诊断与审计包导出。")
else:
    package_manifest = {
        "run_root": str(run_root),
        "profile": globals().get("ACTIVE_WORKFLOW_PROFILE", "paper_full_cuda_formal"),
        "signoff_profile": globals().get("ACTIVE_SIGNOFF_PROFILE", "paper"),
        "stage_status": globals().get("STAGE_STATUS", "<absent>"),
        "required_files": [str(path.relative_to(run_root)) for path in required_files],
        "required_support_files": [
            str(path.relative_to(run_root)) if str(path).startswith(str(run_root)) else path.name
            for path in required_support_files
        ],
        "git_info_file": git_info_path.name if git_info_path is not None else "<absent>",
        "excluded_prefixes": ["artifacts/s2_attention_probe/"],
    }
    manifest_path = artifacts_dir / "package_manifest.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(package_manifest, f, indent=2, ensure_ascii=False)
    print(f"✓ 已写入打包清单：{manifest_path}")

    def _iter_project_export_files():
        scoped_dirs = [
            (records_dir, "records"),
            (artifacts_dir, "artifacts"),
            (outputs_dir, "outputs"),
            (logs_dir, "logs"),
        ]
        for base_dir, prefix in scoped_dirs:
            if not base_dir.exists():
                continue
            for file_path in sorted(base_dir.rglob("*")):
                if not file_path.is_file():
                    continue
                relative_path = file_path.relative_to(base_dir).as_posix()
                arcname = f"{prefix}/{relative_path}"
                if arcname.startswith("artifacts/s2_attention_probe/"):
                    continue
                yield file_path, arcname

        if 'ATTESTATION_INFO_PATH' in globals():
            attestation_path = Path(ATTESTATION_INFO_PATH)
            if attestation_path.exists() and attestation_path.is_file():
                yield attestation_path, f"runtime_metadata/{attestation_path.name}"

        if git_info_path is not None and git_info_path.exists() and git_info_path.is_file():
            yield git_info_path, f"runtime_metadata/{git_info_path.name}"

    archive_name = _build_timestamped_filename(
        f"{CONFIG_CHOICE}_Succeed",
        ".zip",
    )
    archive_path = WORK_DIR / archive_name
    seen_arcnames = set()

    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path, arcname in _iter_project_export_files():
            if arcname in seen_arcnames:
                continue
            seen_arcnames.add(arcname)
            zf.write(file_path, arcname=arcname)

    PROJECT_RESULTS_ARCHIVE_PATH = archive_path
    PROJECT_EXPORT_FILE_COUNT = len(seen_arcnames)
    PROJECT_EXPORT_STATUS = "archive_created"

    synced_archive_path = _sync_file_to_google_drive(archive_path)
    if synced_archive_path is not None:
        PROJECT_RESULTS_ARCHIVE_SYNC_PATH = synced_archive_path
        print(f"✓ 已同步到 Google Drive：{synced_archive_path}")
    else:
        print("ℹ 未检测到 Google Drive 同步目录，跳过同步")

    print("✓ 项目结果打包完成")
    print(f"  文件数量：{PROJECT_EXPORT_FILE_COUNT}")
    print(f"  压缩包：{archive_path}")

    if 'IN_COLAB' in globals() and IN_COLAB:
        try:
            from google.colab import files as colab_files
            colab_files.download(str(archive_path))
            PROJECT_EXPORT_DOWNLOADED = True
            PROJECT_EXPORT_STATUS = "download_triggered"
            print(f"✓ 已触发下载：{archive_path.name}")
        except Exception as exc:
            print(f"⚠ 自动下载失败，请手动下载：{archive_path}，原因：{exc}")
    else:
        print(f"ℹ 非 Colab 环境，请手动获取：{archive_path}")

## 第 6 步：验证工作流输出

检查工作流是否成功生成了所有必要的输出文件。

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

print("=" * 80)
print("验证 formal workflow 输出")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再验证输出")

records_dir = RUN_ROOT / "records"
artifacts_dir = RUN_ROOT / "artifacts"
summary_path = artifacts_dir / "workflow_acceptance" / "paper_full_formal_summary.json"
signoff_rel = "signoff/signoff_report.json"
signoff_report_path = artifacts_dir / signoff_rel

def _load_json_dict(path: Path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    return payload if isinstance(payload, dict) else {}

print("\n[1/6] 检查记录文件...")
expected_records = [
    "embed_record.json",
    "detect_record.json",
    "calibration_record.json",
    "evaluate_record.json",
]

records_found = {}
for record_name in expected_records:
    record_path = records_dir / record_name
    exists = record_path.exists()
    records_found[record_name] = exists
    print(f"  {'✓' if exists else '✗'} {record_name}")

print("\n[2/6] 检查 formal 产物文件...")
expected_artifacts = [
    "evaluation_report.json",
    "run_closure.json",
    signoff_rel,
    "workflow_acceptance/paper_full_formal_summary.json",
    "attestation/attestation_statement.json",
    "attestation/attestation_bundle.json",
    "attestation/attestation_result.json",
    "thresholds/thresholds_artifact.json",
    "../outputs/experiment_matrix/artifacts/grid_summary.json",
]

artifact_paths = {}
artifacts_found = {}
for artifact_name in expected_artifacts:
    artifact_path = (artifacts_dir / artifact_name).resolve() if artifact_name.startswith("../") else artifacts_dir / artifact_name
    artifact_paths[artifact_name] = artifact_path
    artifacts_found[artifact_name] = artifact_path.exists()

missing_artifacts = [name for name, exists in artifacts_found.items() if not exists]
if missing_artifacts and missing_artifacts == [signoff_rel]:
    print("  ⚠ 仅缺 signoff/signoff_report.json，尝试自动补跑 signoff...")
    if 'CEG_WM_ROOT' in globals():
        signoff_profile = globals().get("ACTIVE_SIGNOFF_PROFILE", "paper")
        repo_root = Path(CEG_WM_ROOT)
        signoff_cmd = [
            sys.executable,
            str(repo_root / "scripts" / "run_freeze_signoff.py"),
            "--run-root",
            str(RUN_ROOT),
            "--repo-root",
            str(repo_root),
            "--signoff-profile",
            str(signoff_profile),
        ]
        signoff_result = subprocess.run(
            signoff_cmd,
            cwd=str(repo_root),
            check=False,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
)
        print(f"  signoff return_code: {signoff_result.returncode}")
        for artifact_name, artifact_path in artifact_paths.items():
            artifacts_found[artifact_name] = artifact_path.exists()
    else:
        print("  ✗ 缺少 CEG_WM_ROOT，无法自动补跑 signoff")

for artifact_name in expected_artifacts:
    exists = artifacts_found.get(artifact_name, False)
    print(f"  {'✓' if exists else '✗'} {artifact_name}")

print("\n[3/6] formal summary 语义检查...")
formal_summary = _load_json_dict(summary_path)
formal_summary_ok = False
environment_blocked = None
if formal_summary:
    environment_blocked = formal_summary.get("environment_blocked")
    formal_summary_ok = all([
        formal_summary.get("profile_role") == "paper_full_cuda_formal",
        formal_summary.get("formal_output_expectation_ok") is True,
        formal_summary.get("attestation_bundle_generated") is True,
        formal_summary.get("signoff_decision") == "ALLOW_FREEZE",
        environment_blocked is False,
    ])
    print(f"  profile_role: {formal_summary.get('profile_role')}")
    print(f"  environment_blocked: {environment_blocked}")
    print(f"  formal_output_expectation_ok: {formal_summary.get('formal_output_expectation_ok')}")
    print(f"  attestation_bundle_generated: {formal_summary.get('attestation_bundle_generated')}")
    print(f"  signoff_decision: {formal_summary.get('signoff_decision')}")
else:
    print("  ✗ formal summary 缺失")

print("\n[4/6] layered attestation 检查...")
detect_record = _load_json_dict(records_dir / "detect_record.json")
attestation_payload = detect_record.get("attestation", {}) if isinstance(detect_record, dict) else {}
authenticity_result = attestation_payload.get("authenticity_result", {}) if isinstance(attestation_payload, dict) else {}
image_evidence_result = attestation_payload.get("image_evidence_result", {}) if isinstance(attestation_payload, dict) else {}
final_attested_decision = attestation_payload.get("final_event_attested_decision", {}) if isinstance(attestation_payload, dict) else {}
bundle_verification = attestation_payload.get("bundle_verification", {}) if isinstance(attestation_payload, dict) else {}
bundle_status = authenticity_result.get("bundle_status") if isinstance(authenticity_result, dict) else None
bundle_verification_status = bundle_verification.get("status") if isinstance(bundle_verification, dict) else None
is_event_attested = final_attested_decision.get("is_event_attested") if isinstance(final_attested_decision, dict) else None

layered_attestation_ok = all([
    isinstance(authenticity_result, dict) and bool(authenticity_result),
    isinstance(image_evidence_result, dict) and bool(image_evidence_result),
    isinstance(final_attested_decision, dict) and bool(final_attested_decision),
    isinstance(is_event_attested, bool),
    isinstance(bundle_status, str) and bool(bundle_status),
])
if layered_attestation_ok and isinstance(bundle_verification_status, str) and bundle_verification_status:
    layered_attestation_ok = bundle_verification_status == bundle_status
if layered_attestation_ok and bundle_status != "ok":
    layered_attestation_ok = is_event_attested is False
if layered_attestation_ok and bundle_status == "ok":
    layered_attestation_ok = authenticity_result.get("status") in {"authentic", "ok"}

print(f"  authenticity_result_present: {isinstance(authenticity_result, dict) and bool(authenticity_result)}")
print(f"  image_evidence_result_present: {isinstance(image_evidence_result, dict) and bool(image_evidence_result)}")
print(f"  final_event_attested_decision_present: {isinstance(final_attested_decision, dict) and bool(final_attested_decision)}")
print(f"  bundle_status: {bundle_status}")
print(f"  bundle_verification.status: {bundle_verification_status}")
print(f"  is_event_attested: {is_event_attested}")
print(f"  layered_attestation_ok: {layered_attestation_ok}")

print("\n[5/6] evaluation_report 语义锚点检查...")
evaluation_report_path = artifacts_dir / "evaluation_report.json"
semantic_ok = False
if evaluation_report_path.exists():
    raw_eval_report = _load_json_dict(evaluation_report_path)
    eval_report = raw_eval_report.get("evaluation_report", raw_eval_report) if isinstance(raw_eval_report, dict) else {}
    required_anchor_fields = [
        "cfg_digest",
        "plan_digest",
        "thresholds_digest",
        "threshold_metadata_digest",
        "impl_digest",
        "fusion_rule_version",
        "attack_protocol_version",
        "attack_protocol_digest",
        "policy_path",
    ]

    def _read_anchor(report_obj, key):
        value = report_obj.get(key)
        if isinstance(value, str) and value:
            return value
        anchors_obj = report_obj.get("anchors")
        if isinstance(anchors_obj, dict):
            nested_value = anchors_obj.get(key)
            if isinstance(nested_value, str) and nested_value:
                return nested_value
        return None

    missing = [
        field for field in required_anchor_fields
        if (_read_anchor(eval_report, field) is None or _read_anchor(eval_report, field) == "<absent>")
    ]
    report_type_ok = eval_report.get("report_type") == "evaluation_report"
    metrics_list_ok = isinstance(eval_report.get("metrics_by_attack_condition"), list)
    semantic_ok = report_type_ok and metrics_list_ok and (len(missing) == 0)
    print(f"  report_type_ok: {report_type_ok}")
    print(f"  metrics_by_attack_condition_is_list: {metrics_list_ok}")
    print(f"  missing_anchor_fields: {missing}")
else:
    print("  ✗ 缺失 evaluation_report.json，无法进行语义检查")

print("\n[6/6] 结论汇总...")
all_required_records = all(records_found.values())
all_required_artifacts = all(artifacts_found.values())
is_signoff_allow = (_load_json_dict(signoff_report_path).get("freeze_signoff_decision") == "ALLOW_FREEZE")

VALIDATION_SUMMARY = {
    "records_complete": all_required_records,
    "artifacts_complete": all_required_artifacts,
    "formal_summary_ok": formal_summary_ok,
    "layered_attestation_ok": layered_attestation_ok,
    "evaluation_semantic_ok": semantic_ok,
    "signoff_allow": is_signoff_allow,
    "environment_blocked": environment_blocked,
}

REQUIRED_PACKAGE_FILES = [
    records_dir / "embed_record.json",
    records_dir / "detect_record.json",
    records_dir / "calibration_record.json",
    records_dir / "evaluate_record.json",
    artifacts_dir / "evaluation_report.json",
    artifacts_dir / "run_closure.json",
    artifacts_dir / "signoff" / "signoff_report.json",
    artifacts_dir / "workflow_acceptance" / "paper_full_formal_summary.json",
    artifacts_dir / "attestation" / "attestation_statement.json",
    artifacts_dir / "attestation" / "attestation_bundle.json",
    artifacts_dir / "attestation" / "attestation_result.json",
    artifacts_dir / "thresholds" / "thresholds_artifact.json",
    RUN_ROOT / "outputs" / "experiment_matrix" / "artifacts" / "grid_summary.json",
]

print(f"  records 完整：{all_required_records}")
print(f"  artifacts 完整：{all_required_artifacts}")
print(f"  signoff 通过：{is_signoff_allow}")
print(f"  formal summary 合格：{formal_summary_ok}")
print(f"  layered attestation 合格：{layered_attestation_ok}")
print(f"  evaluation_report 语义合格：{semantic_ok}")
print(f"  输出目录：{RUN_ROOT}")

## 第 7 步：失败时快速诊断与审计包导出

如果第 5.1 步检测到缺失必需产物而禁止打包，请先完成第 6 步验证，再执行本步生成诊断与审计包。

如果第 5.1 步已经成功导出完整项目结果，本步只做快速诊断，不再重复触发下载。

In [ ]:
import json
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("第 7 步：快速诊断与按需审计包导出")
print("=" * 80)

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再执行快速诊断")

run_root = Path(RUN_ROOT)
if not run_root.exists():
    raise RuntimeError(f"RUN_ROOT 不存在：{run_root}")

project_export_success = globals().get("PROJECT_EXPORT_STATUS") in {"archive_created", "download_triggered"}
validation_summary = globals().get("VALIDATION_SUMMARY", {})


def _load_json(path: Path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _ensure_git_info_file() -> Path | None:
    if 'GIT_INFO_PATH' in globals():
        candidate_path = Path(GIT_INFO_PATH)
        if candidate_path.exists() and candidate_path.is_file():
            return candidate_path

    if 'CEG_WM_ROOT' not in globals():
        return None

    repo_root = Path(CEG_WM_ROOT)
    git_info_path = WORK_DIR / "git_version_info.json"

    def _run_git(command):
        result = subprocess.run(
            command,
            cwd=str(repo_root),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Git 信息补生成失败，return_code={result.returncode}，command={' '.join(command)}"
            )
        return result.stdout.strip()

    version_info = {
        "repo_url": _run_git(["git", "remote", "get-url", "origin"]),
        "branch": _run_git(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
        "target_dir": str(repo_root),
        "sync_mode": globals().get("sync_mode", "<unknown>"),
        "outputs_cleaned": True,
        "commit_hash": _run_git(["git", "rev-parse", "HEAD"]),
        "commit_hash_short": _run_git(["git", "rev-parse", "--short=8", "HEAD"]),
        "commit_message": _run_git(["git", "log", "-1", "--pretty=%s"]),
        "commit_timestamp": _run_git(["git", "log", "-1", "--pretty=%ci"]),
        "commit_author": _run_git(["git", "log", "-1", "--pretty=%an <%ae>"]),
        "sync_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "generated_for_export": True,
    }

    with open(git_info_path, "w", encoding="utf-8") as f:
        json.dump(version_info, f, indent=2, ensure_ascii=False)

    globals()["GIT_INFO_PATH"] = git_info_path
    print(f"✓ 已补生成 Git 信息：{git_info_path}")
    return git_info_path


def _build_timestamped_filename(base_name: str, suffix: str) -> str:
    timestamp_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{base_name}_{timestamp_suffix}{suffix}"


def _sync_file_to_google_drive(source_path: Path) -> Path | None:
    drive_export_dir = globals().get("GOOGLE_DRIVE_EXPORT_DIR")
    if drive_export_dir is None:
        return None

    drive_export_dir = Path(drive_export_dir)
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    target_path = drive_export_dir / source_path.name
    shutil.copy2(source_path, target_path)
    return target_path


git_info_error = None
try:
    git_info_path = _ensure_git_info_file()
except Exception as exc:
    git_info_path = None
    git_info_error = str(exc)

summary_path = run_root / "artifacts" / "workflow_acceptance" / "paper_full_formal_summary.json"
detect_record_path = run_root / "records" / "detect_record.json"
signoff_report_path = run_root / "artifacts" / "signoff" / "signoff_report.json"
grid_summary_path = run_root / "outputs" / "experiment_matrix" / "artifacts" / "grid_summary.json"
workflow_log_path = run_root / "logs" / "workflow_execution.log"

summary_obj = _load_json(summary_path)
detect_record = _load_json(detect_record_path)
signoff_report = _load_json(signoff_report_path)
grid_summary = _load_json(grid_summary_path)

failed_step = None
workflow_return_code = None
if workflow_log_path.exists():
    with open(workflow_log_path, "r", encoding="utf-8") as f:
        log_lines = f.read().splitlines()
    for idx, line in enumerate(log_lines):
        if line.strip() == "RETURN_CODE:" and idx + 1 < len(log_lines):
            try:
                workflow_return_code = int(log_lines[idx + 1].strip())
            except Exception:
                workflow_return_code = None
        if "step=" in line and "return_code=" in line:
            try:
                step_name = line.split("step=", 1)[1].split()[0]
                step_rc = int(line.split("return_code=", 1)[1].split()[0])
            except Exception:
                continue
            if step_rc != 0:
                failed_step = step_name

print(f"项目结果包状态：{globals().get('PROJECT_EXPORT_STATUS', '<absent>')}")
print(f"完整项目结果包：{globals().get('PROJECT_RESULTS_ARCHIVE_PATH', '<absent>')}")
print(f"是否已成功导出完整项目结果：{project_export_success}")
print(f"Git 信息文件：{git_info_path if git_info_path is not None else '<absent>'}")
if git_info_error:
    print(f"Git 信息补生成错误：{git_info_error}")

print("\n[1/4] formal summary 状态")
if summary_obj:
    print(f"  profile_role: {summary_obj.get('profile_role')}")
    print(f"  workflow_exit_code: {summary_obj.get('workflow_exit_code')}")
    print(f"  environment_blocked: {summary_obj.get('environment_blocked')}")
    print(f"  formal_output_expectation_ok: {summary_obj.get('formal_output_expectation_ok')}")
    print(f"  attestation_bundle_generated: {summary_obj.get('attestation_bundle_generated')}")
    print(f"  signoff_decision: {summary_obj.get('signoff_decision')}")
else:
    print("  ✗ formal summary 缺失")

print("\n[2/4] layered attestation 状态")
attestation_payload = detect_record.get("attestation", {}) if isinstance(detect_record, dict) else {}
authenticity_result = attestation_payload.get("authenticity_result", {}) if isinstance(attestation_payload, dict) else {}
image_evidence_result = attestation_payload.get("image_evidence_result", {}) if isinstance(attestation_payload, dict) else {}
final_attested_decision = attestation_payload.get("final_event_attested_decision", {}) if isinstance(attestation_payload, dict) else {}
bundle_verification = attestation_payload.get("bundle_verification", {}) if isinstance(attestation_payload, dict) else {}

print(f"  authenticity_result_present: {isinstance(authenticity_result, dict) and bool(authenticity_result)}")
print(f"  image_evidence_result_present: {isinstance(image_evidence_result, dict) and bool(image_evidence_result)}")
print(f"  final_event_attested_decision_present: {isinstance(final_attested_decision, dict) and bool(final_attested_decision)}")
print(f"  bundle_status: {authenticity_result.get('bundle_status', '<absent>')}")
print(f"  bundle_verification.status: {bundle_verification.get('status', '<absent>')}")
print(f"  is_event_attested: {final_attested_decision.get('is_event_attested', '<absent>')}")

print("\n[3/4] workflow / signoff / validation 状态")
print(f"  workflow_return_code: {workflow_return_code}")
print(f"  failed_step: {failed_step}")
print(f"  freeze_signoff_decision: {signoff_report.get('freeze_signoff_decision', '<absent>')}")
print(f"  validation_summary: {validation_summary if validation_summary else '<absent>'}")
if isinstance(grid_summary, dict) and grid_summary:
    print(f"  grid_summary keys: {sorted(grid_summary.keys())[:8]}")
else:
    print("  grid_summary: <absent>")

print("\n[4/4] 生成诊断与审计包")
diagnosis_lines = [
    "=" * 80,
    "CEG-WM Formal Workflow Diagnosis",
    "=" * 80,
    f"generated_at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"run_root: {run_root}",
    f"stage_status: {globals().get('STAGE_STATUS', '<absent>')}",
    f"project_export_status: {globals().get('PROJECT_EXPORT_STATUS', '<absent>')}",
    f"project_results_archive_path: {globals().get('PROJECT_RESULTS_ARCHIVE_PATH', '<absent>')}",
    f"git_info_path: {git_info_path if git_info_path is not None else '<absent>'}",
    f"git_info_error: {git_info_error if git_info_error else '<absent>'}",
    f"workflow_return_code: {workflow_return_code}",
    f"failed_step: {failed_step}",
    "",
    "[validation_summary]",
    json.dumps(validation_summary, indent=2, ensure_ascii=False) if validation_summary else "<absent>",
    "",
    "[formal_summary]",
    json.dumps(summary_obj, indent=2, ensure_ascii=False) if summary_obj else "<absent>",
    "",
    "[detect_attestation]",
    json.dumps(attestation_payload, indent=2, ensure_ascii=False) if attestation_payload else "<absent>",
]

diagnosis_path = run_root / "logs" / "DIAGNOSIS.txt"
diagnosis_path.parent.mkdir(parents=True, exist_ok=True)
with open(diagnosis_path, "w", encoding="utf-8") as f:
    f.write("\n".join(diagnosis_lines))

zip_name = _build_timestamped_filename(f"{CONFIG_CHOICE}_Diagnostic", ".zip")
zip_path = WORK_DIR / zip_name
seen_arcnames = set()


def _write_unique_file(zf, full_path: Path, arcname: str):
    if arcname in seen_arcnames:
        return
    if not full_path.exists() or not full_path.is_file():
        return
    seen_arcnames.add(arcname)
    zf.write(full_path, arcname=arcname)


with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for base_dir, prefix in [
        (run_root / "records", "records"),
        (run_root / "artifacts", "artifacts"),
        (run_root / "outputs", "outputs"),
        (run_root / "logs", "logs"),
    ]:
        if not base_dir.exists():
            continue
        for file_path in sorted(base_dir.rglob("*")):
            if not file_path.is_file():
                continue
            relative_path = file_path.relative_to(base_dir).as_posix()
            arcname = f"{prefix}/{relative_path}"
            if arcname.startswith("artifacts/s2_attention_probe/"):
                continue
            _write_unique_file(zf, file_path, arcname)

    if 'ATTESTATION_INFO_PATH' in globals():
        attestation_info_path = Path(ATTESTATION_INFO_PATH)
        _write_unique_file(zf, attestation_info_path, f"runtime_metadata/{attestation_info_path.name}")

    if git_info_path is not None:
        _write_unique_file(zf, git_info_path, f"runtime_metadata/{git_info_path.name}")

DIAGNOSTIC_ARCHIVE_PATH = zip_path
DIAGNOSTIC_ARCHIVE_SYNC_PATH = _sync_file_to_google_drive(zip_path)
print(f"✓ 诊断包生成：{zip_path}")
print(f"  去重后文件数量：{len(seen_arcnames)}")
if DIAGNOSTIC_ARCHIVE_SYNC_PATH is not None:
    print(f"✓ 已同步到 Google Drive：{DIAGNOSTIC_ARCHIVE_SYNC_PATH}")
else:
    print("ℹ 未检测到 Google Drive 同步目录，跳过同步")

if project_export_success:
    print("ℹ 第 5.1 步已经成功导出完整项目结果，本步不再重复触发下载。")
    print(f"  如需查看诊断包，可手动获取：{zip_path}")
else:
    if 'IN_COLAB' in globals() and IN_COLAB and colab_files is not None:
        try:
            colab_files.download(str(zip_path))
            print(f"✓ 已触发审计包下载：{zip_path.name}")
        except Exception as exc:
            print(f"⚠ 自动下载失败，请手动下载：{zip_path}，原因：{exc}")
    else:
        print(f"ℹ 非 Colab 环境，请手动获取：{zip_path}")

## 配置选项参考

本 Notebook 第 5 步直接使用项目内的 `configs/paper_full_cuda.yaml`，并通过正式 GPU 验收脚本执行。

| 配置项 | 值 | 说明 |
|---------|------|------|
| `verification entry` | `scripts/run_paper_full_workflow_verification.py` | 官方 formal GPU 验收入口，内部复用 onefile 主链 |
| `cfg` | `configs/paper_full_cuda.yaml` | 仓库内正式 paper 配置（不再生成临时 Colab yaml） |
| `model_id` | `stabilityai/stable-diffusion-3.5-medium` | 真实 SD3.5 模型 |
| `device` | `cuda` | 要求 Colab GPU Runtime |
| `signoff profile` | `paper` | 启用严格审计与签署路径 |
| `formal summary` | `artifacts/workflow_acceptance/paper_full_formal_summary.json` | 区分环境阻断与代码失败，并固定 formal 输出状态 |

### 如需调整
- 如需修改模型或参数，请直接在项目配置文件中调整，或新建正式配置文件后在第 5 步改 `PROJECT_PAPER_CFG`。